# MATS Simplex Take-Home: Exploratory Analysis
Non-ergodic Mess3 mixture — geometry, probing, and subspace overlap.

In [ ]:
import sys, os

%pip install "numpy<2.0" transformer_lens scikit-learn matplotlib tqdm -q

if not os.path.exists('/content/MATS-take-home'):
    !git clone https://github.com/vedantgaur/MATS-take-home.git
else:
    %cd /content/MATS-take-home && !git pull && %cd /content

sys.path.append('/content/MATS-take-home')

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformer_lens import HookedTransformer

from data.mess3_generator import Mess3Process, NonErgodicMess3Dataset
from models.train import get_toy_config, train_model
from analysis.geometry import extract_activations, calculate_cev, plot_cev, plot_2d_pca, extract_all_layers
from analysis.orthogonality import compare_all_processes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 1. Data Generation

In [ ]:
p1 = Mess3Process(alpha=0.60, x=0.15)  # Process 1
p2 = Mess3Process(alpha=0.79, x=0.11)  # Process 2 — similar dynamics to P1
p3 = Mess3Process(alpha=0.60, x=0.50)  # Process 3 — y=0, very different entropy

processes = [p1, p2, p3]
seq_length = 16

train_dataset = NonErgodicMess3Dataset(num_samples=20000, seq_length=seq_length, processes=processes)
val_dataset   = NonErgodicMess3Dataset(num_samples=1000,  seq_length=seq_length, processes=processes)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)

print(f'Train: {len(train_dataset)} samples | Val: {len(val_dataset)} samples')
print(f'Sequence length: {seq_length} | Vocab size: 3')

## 2. Training

In [ ]:
cfg   = get_toy_config(vocab_size=3, d_model=64, n_ctx=seq_length)
model = HookedTransformer(cfg).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {total_params:,}')

loss_history = train_model(model, train_loader, epochs=150, lr=1e-3)

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Cross Entropy Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Final loss: {loss_history[-1]:.4f}')

## 3. Residual Stream Geometry (Layer-wise)

For each layer: (a) CEV to measure effective dimensionality, (b) PCA scatter colored by process ID,
(c) subspace overlap heatmap with numeric scores.

In [ ]:
print('Extracting activations across all layers...')
acts_by_layer, labels = extract_all_layers(model, val_loader)
n_layers = len(acts_by_layer)

# Store PCA models for later use
pca_models = {}

for layer_idx in range(n_layers):
    print(f'\n{"="*50}')
    print(f'LAYER {layer_idx}')
    print(f'{"="*50}')

    acts = acts_by_layer[layer_idx]

    # --- CEV ---
    pca_model, cev = calculate_cev(acts)
    pca_models[layer_idx] = pca_model
    dims_95 = np.argmax(cev >= 0.95) + 1
    print(f'  Dims for 95% variance: {dims_95}  (theoretical optimal: 8)')

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # CEV plot
    axes[0].plot(range(1, len(cev)+1), cev, marker='o', markersize=3, lw=1)
    axes[0].axhline(0.95, color='r', ls='--', label='95%')
    axes[0].axvline(dims_95, color='g', ls='--', label=f'{dims_95} dims')
    axes[0].axvline(8, color='orange', ls=':', label='Theoretical (8)')
    axes[0].set_title(f'Layer {layer_idx}: CEV')
    axes[0].set_xlabel('# Principal Components')
    axes[0].set_ylabel('Cumulative Explained Variance')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # PC1 vs PC2
    proj = pca_model.transform(acts)
    sc = axes[1].scatter(proj[:, 0], proj[:, 1], c=labels, cmap='viridis', alpha=0.6, s=10)
    plt.colorbar(sc, ax=axes[1], label='Process ID')
    axes[1].set_title(f'Layer {layer_idx}: PC1 vs PC2 (macroscopic)')
    axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

    # PC3 vs PC4
    sc2 = axes[2].scatter(proj[:, 2], proj[:, 3], c=labels, cmap='viridis', alpha=0.6, s=10)
    plt.colorbar(sc2, ax=axes[2], label='Process ID')
    axes[2].set_title(f'Layer {layer_idx}: PC3 vs PC4 (microscopic)')
    axes[2].set_xlabel('PC3'); axes[2].set_ylabel('PC4')

    plt.tight_layout()
    plt.show()

    # --- Subspace Overlap ---
    overlap_matrix = compare_all_processes(acts, labels, k_dims=2)
    process_names = ['Process 1', 'Process 2', 'Process 3']

    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(overlap_matrix, cmap='magma', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, label='Overlap (0=Orthogonal, 1=Parallel)')
    ax.set_xticks([0,1,2]); ax.set_xticklabels(process_names, rotation=20)
    ax.set_yticks([0,1,2]); ax.set_yticklabels(process_names)
    ax.set_title(f'Layer {layer_idx}: Subspace Overlap')

    # Annotate with numeric values
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{overlap_matrix[i,j]:.3f}',
                    ha='center', va='center',
                    color='white' if overlap_matrix[i,j] < 0.6 else 'black',
                    fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'  Overlap matrix (numeric):')
    print(f'    P1-P2: {overlap_matrix[0,1]:.4f}')
    print(f'    P1-P3: {overlap_matrix[0,2]:.4f}')
    print(f'    P2-P3: {overlap_matrix[1,2]:.4f}')

## 4. Context Position Evolution

Project activations at different sequence positions onto the last-layer PCA basis.
Uses the **full** validation set (not just one batch) for reliable geometry.

In [ ]:
def extract_all_positions(model, dataloader):
    """Extract residual stream activations at ALL sequence positions."""
    model.eval()
    last_layer = model.cfg.n_layers - 1
    hook_name  = f'blocks.{last_layer}.hook_resid_post'

    all_acts   = []   # will be (N, seq_len, d_model)
    all_labels = []

    with torch.no_grad():
        for batch_x, _, labs in dataloader:
            batch_x = batch_x.to(model.cfg.device)
            _, cache = model.run_with_cache(batch_x, names_filter=[hook_name])
            all_acts.append(cache[hook_name].cpu().numpy())
            all_labels.append(labs.numpy())

    return np.concatenate(all_acts, axis=0), np.concatenate(all_labels)


print('Extracting full-sequence activations from val set...')
all_pos_acts, pos_labels = extract_all_positions(model, val_loader)
print(f'Shape: {all_pos_acts.shape}  (samples, seq_len, d_model)')

# Use the last-layer PCA (fit on last-position acts) as the fixed projection basis
final_pca = pca_models[n_layers - 1]

positions_to_plot = [0, 3, 7, 14]  # l = 1, 4, 8, 15
fig, axes = plt.subplots(1, len(positions_to_plot), figsize=(20, 5), sharex=True, sharey=True)

for i, pos in enumerate(positions_to_plot):
    pos_acts = all_pos_acts[:, pos, :]       # (N, d_model)
    proj     = final_pca.transform(pos_acts) # project onto shared PCA basis

    sc = axes[i].scatter(proj[:, 0], proj[:, 1], c=pos_labels,
                         cmap='viridis', alpha=0.7, s=12)
    axes[i].set_title(f'Position l={pos+1}')
    axes[i].set_xlabel('PC 1')
    if i == 0:
        axes[i].set_ylabel('PC 2')

plt.suptitle('Activation Geometry Evolution Across Context Window', fontsize=14)
plt.colorbar(sc, ax=axes[-1], label='Process ID')
plt.tight_layout()
plt.show()

## 5. Linear Probe: Residual Stream → Belief States

**Correct implementation**: compute ground-truth HMM belief states via the forward algorithm,
then regress from activations to those belief states.

For each sequence generated by process k, the belief state η_k evolves with the tokens.
The inactive processes stay at the stationary distribution [1/3, 1/3, 1/3].
Target is the concatenated 9-dim vector [η_1, η_2, η_3], representing the global predictive state.

In [ ]:
def compute_belief_trajectory(token_sequence, process: Mess3Process):
    """
    Run the HMM forward algorithm to compute the belief state at each position.
    Returns array of shape (seq_len, 3), where each row is a probability distribution
    over the 3 hidden states after observing tokens 0..t.
    """
    T_matrices = [process.T0, process.T1, process.T2]
    eta = process.state_dist.copy()   # start at stationary: [1/3, 1/3, 1/3]
    trajectory = []

    for token in token_sequence:
        T_x = T_matrices[int(token)]
        # Bayes update: eta' = eta @ T_x, then normalize
        eta = eta @ T_x
        eta = eta / eta.sum()
        trajectory.append(eta.copy())

    return np.array(trajectory)   # (seq_len, 3)


def build_belief_targets(val_dataset, processes):
    """
    For each sample, build the 9-dim ground-truth belief vector at the LAST position.
    Inactive processes stay at stationary [1/3, 1/3, 1/3].
    Returns array of shape (N, 9).
    """
    stationary = np.array([1/3, 1/3, 1/3])
    n_proc = len(processes)
    targets = []

    for idx in range(len(val_dataset)):
        # x is seq[:-1], y is seq[1:], label is process_idx
        x, _, label = val_dataset[idx]
        label = int(label)

        # x has length seq_len-1; we want the belief after seeing all of x
        active_trajectory = compute_belief_trajectory(x.numpy(), processes[label])
        active_belief_at_last = active_trajectory[-1]  # shape (3,)

        # Concatenate: active process gets real belief, others get stationary
        full_belief = []
        for k in range(n_proc):
            if k == label:
                full_belief.append(active_belief_at_last)
            else:
                full_belief.append(stationary)

        targets.append(np.concatenate(full_belief))  # (9,)

    return np.array(targets)  # (N, 9)


print('Computing ground-truth belief states...')
belief_targets = build_belief_targets(val_dataset, processes)
print(f'Belief target shape: {belief_targets.shape}  (samples, 9)')
print(f'Sample belief vector: {belief_targets[0].round(3)}')

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Use last-layer, last-position activations
acts_final = acts_by_layer[n_layers - 1]   # (N, 64)

X_train, X_test, y_train, y_test = train_test_split(
    acts_final, belief_targets, test_size=0.2, random_state=42
)

# Cross-validated alpha selection
from sklearn.linear_model import RidgeCV
probe = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
probe.fit(X_train, y_train)

y_pred = probe.predict(X_test)
r2_overall = r2_score(y_test, y_pred)

# Per-process R² (each process contributes a 3-dim block)
r2_per_process = []
for k in range(len(processes)):
    start, end = k*3, k*3 + 3
    r2_k = r2_score(y_test[:, start:end], y_pred[:, start:end])
    r2_per_process.append(r2_k)

print(f'Linear Probe Results (correct belief state targets):')
print(f'  Overall R²:    {r2_overall:.4f}')
for k, r2_k in enumerate(r2_per_process):
    print(f'  Process {k+1} R²: {r2_k:.4f}')
print(f'  Best Ridge alpha: {probe.alpha_}')

# Sanity check: random baseline
random_acts = np.random.randn(*acts_final.shape)
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(random_acts, belief_targets, test_size=0.2, random_state=42)
random_probe = Ridge(alpha=1.0).fit(X_tr_r, y_tr_r)
r2_random = r2_score(y_te_r, random_probe.predict(X_te_r))
print(f'  Random baseline R²: {r2_random:.4f}')

## 6. Embedding Matrix SVD

Shai et al. find factorization begins at the embedding matrix, visible as a sharp rank collapse
in the singular value spectrum. We check whether our model shows the same signature.

In [ ]:
# Extract the token embedding matrix W_E: shape (vocab_size, d_model) = (3, 64)
W_E = model.embed.W_E.detach().cpu().numpy()
print(f'W_E shape: {W_E.shape}')

U, S, Vt = np.linalg.svd(W_E, full_matrices=False)

print(f'Singular values: {S.round(4)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Singular value spectrum
axes[0].bar(range(len(S)), S, color='steelblue')
axes[0].set_title('Singular Value Spectrum of W_E')
axes[0].set_xlabel('Singular Value Index')
axes[0].set_ylabel('Singular Value')
axes[0].grid(True, alpha=0.3)

# Token embeddings projected onto top 2 singular vectors
# U has shape (vocab_size, min_dim) = (3, 3)
# Color by token identity
token_proj = W_E @ Vt.T   # (3, 3): project tokens onto right singular vectors
for tok_id in range(W_E.shape[0]):
    axes[1].scatter(token_proj[tok_id, 0], token_proj[tok_id, 1],
                    s=200, label=f'Token {tok_id}', zorder=5)
    axes[1].annotate(f'  {tok_id}', (token_proj[tok_id, 0], token_proj[tok_id, 1]))
axes[1].set_title('Token Embeddings: SV1 vs SV2')
axes[1].set_xlabel('Right Singular Vector 1')
axes[1].set_ylabel('Right Singular Vector 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Note: vocab_size=3, so W_E is (3 x 64). Max rank is 3.
# The theoretical Mess3 embedding should use 2 dims (2-simplex).
# Check if the 3rd singular value is near-zero (low-rank factored structure).
print(f'\nEffective rank check:')
print(f'  SV1: {S[0]:.4f}, SV2: {S[1]:.4f}, SV3: {S[2]:.4f}')
print(f'  Ratio SV3/SV1: {S[2]/S[0]:.4f}  (near-zero → low-rank factored structure)')

## 7. Overlap Evolution Summary

Plot all pairwise overlap scores across layers side by side for easy comparison.

In [ ]:
overlap_by_layer = {}
for layer_idx in range(n_layers):
    acts = acts_by_layer[layer_idx]
    overlap_by_layer[layer_idx] = compare_all_processes(acts, labels, k_dims=2)

pairs = [('P1-P2', 0, 1), ('P1-P3', 0, 2), ('P2-P3', 1, 2)]
layer_ids = list(range(n_layers))

plt.figure(figsize=(8, 4))
for name, i, j in pairs:
    scores = [overlap_by_layer[l][i, j] for l in layer_ids]
    plt.plot(layer_ids, scores, marker='o', linewidth=2, label=name)

plt.xlabel('Layer')
plt.ylabel('Subspace Overlap (0=Orthogonal)')
plt.title('Pairwise Subspace Overlap Across Layers')
plt.xticks(layer_ids)
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\nNumerical overlap scores by layer:')
print(f'{"Layer":<8}', '  '.join(f'{n:<10}' for n, *_ in pairs))
for l in layer_ids:
    row = [overlap_by_layer[l][i, j] for _, i, j in pairs]
    print(f'{l:<8}', '  '.join(f'{v:<10.4f}' for v in row))